# MACE+Graph2Mat

This notebook will show you how to integrate a `MACE` model with `Graph2Mat` through the python API. Note that you can also use `MACE+Graph2Mat` through the Command Line Interface (CLI).

Prerequisites
-------------
Before reading this notebook, **make sure you have read the [notebook on computing a matrix](<./Computing a matrix.ipynb>) and [the notebook on batching](./Batching.ipynb)**, which introduce the basic concepts of `graph2mat` that we are going to assume are already known. Also **we will use exactly the same setup as in the batching notebook**, with the only difference that we will add target matrices to each structure.

In [1]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [2]:
import numpy as np
import pandas as pd
import torch

# To load plotly templates for sisl visualization
import sisl.viz

from e3nn import o3

from graph2mat import (
    BasisConfiguration,
    PointBasis,
    BasisTableWithEdges,
    MatrixDataProcessor,
)
from graph2mat.bindings.torch import TorchBasisMatrixDataset, TorchBasisMatrixData

from graph2mat.bindings.e3nn import E3nnGraph2Mat

from graph2mat.tools.viz import plot_basis_matrix


from torch_geometric.loader import DataLoader

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._show_config()


Generating a dataset
--------------------

We generate a dataset here just as we have done in the other notebooks.

In [3]:
# The basis
point_1 = PointBasis("A", R=2, basis="0e", basis_convention="spherical")
point_2 = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical")


# The basis table.
table = BasisTableWithEdges([point_1, point_2])

# The data processor.
processor = MatrixDataProcessor(
    basis_table=table, symmetric_matrix=False,  # Matrix is not square
    sub_point_matrix=False
)

positions = np.array([[0, 0, 0], [6.0, 0, 0], [9.0, 0, 0]])  # positions of the points

config1 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions,
    basis=[point_1, point_2],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

dataset = TorchBasisMatrixDataset([config1], data_processor=processor)


loader = DataLoader(dataset, batch_size=1)

data = next(iter(loader))

BasisTableWithEdges: is_square = True
BasisTableWithEdges: all matrix roles = [None, None]
Basis sizes: [1 5]
Basis sizes: [1 5]
In BasisTableWithEdges: 
self.edge_type == point_types_to_edge_types:
[[ 0  1]
 [-1  2]]
self.point_block_shape:
[[1 5]
 [1 5]]
self.point_block_size:
[ 1 25]
Point type to edge type:
[[ 0  1]
 [-1  2]]
Edge type to point types:
[[0 0]
 [0 1]
 [1 1]]
Edge block shape:
[[1 1 5]
 [1 5 5]]
Edge block size:
[ 1  5 25]
self.basis_table.R is an array: [2 5]
point_types: [0 1 0]
self.basis_table.R[point_types]: [2 5 2]
Cutoff: [1.9999 4.9999 1.9999]
In BasisMatrixData.from_config 1:
edge_index: [[0 1 1 2]
 [1 0 2 1]]
edge_types: [ 1 -1 -1  1]
In sort_edge_index:
isc_off: [[[0]]]
sc_shifts: [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
isc: [0 0 0 0]
In BasisMatrixData.from_config 2:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]
In BasisMatrixData.from_config 3:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]


Initializing a MACE model
-------------------------

We will now initialize a normal MACE model.

Note that you must have MACE installed, which you can do with:

```
pip install mace_torch
```

In [4]:
from mace.modules import MACE, RealAgnosticResidualInteractionBlock

num_interactions = 3
hidden_irreps = o3.Irreps("1x0e + 1x1o")

mace_model = MACE(
    r_max=10,
    num_bessel=10,
    num_polynomial_cutoff=10,
    max_ell=2,  # 1,
    interaction_cls=RealAgnosticResidualInteractionBlock,
    interaction_cls_first=RealAgnosticResidualInteractionBlock,
    num_interactions=num_interactions,
    num_elements=2,
    hidden_irreps=hidden_irreps,
    MLP_irreps=o3.Irreps("2x0e"),
    atomic_energies=torch.tensor([0, 0]),
    avg_num_neighbors=2,
    atomic_numbers=[0, 1],
    correlation=2,
    gate=None,
)

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/mace/modules/blocks.py:187: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/to

Now, we can pass our data through the mace model. MACE outputs many things, but we are just interested in the node features, which we can get from the `"node_feats"` key.

In [5]:
mace_output = mace_model(data)
mace_output["node_feats"]

tensor([[-1.5024e-02,  0.0000e+00,  0.0000e+00, -1.9594e-05,  1.4302e-02,
          0.0000e+00,  0.0000e+00,  1.3261e-02, -2.6200e-03],
        [-4.1464e-01,  0.0000e+00,  0.0000e+00, -4.5309e-06, -2.6564e-01,
          0.0000e+00,  0.0000e+00, -4.5035e-05,  2.7080e-02],
        [-1.4325e-02,  0.0000e+00,  0.0000e+00, -3.5213e-05,  1.3495e-02,
          0.0000e+00,  0.0000e+00,  3.0715e-03, -3.2025e-03]],
       grad_fn=<CatBackward0>)

Our `Graph2Mat` model will take these node features and convert them to a matrix. Therefore we need to know what its irreps are, and then initialize the `Graph2Mat` module.

In [6]:
# MACE outputs as node features the hidden irreps for each interaction, except
# in the last interaction, where it computes just scalar features.
mace_out_irreps = hidden_irreps * (num_interactions - 1) + str(hidden_irreps[0])

# Initialize the matrix model with this information
matrix_model = E3nnGraph2Mat(
    unique_basis=table,
    irreps=dict(node_feats_irreps=mace_out_irreps),
    symmetric=False,  # Matrix is not square
    # We would need to also implement passing the edge information in order to use
    # preprocessing_edges. As shown later, graph2mat can do this automatically for you.
    preprocessing_edges=None,
)

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/ICN2/snavarro

Now, we can use the matrix model, passing the node features computed by MACE:

In [7]:
node_labels, edge_labels = matrix_model(data=data, node_feats=mace_output["node_feats"])

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0]
shapes:  [[1 5]
 [1 5]]
shapes_inv:  [[1 5]
 [1 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26  1]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True, False,  True])
j_edges: (~i_edges) tensor([ True, False,  True, False])
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [ 1 -1  1 -1]
shapes:  [[1 1 5]
 [1 5 5]]
shapes_inv:  [[1 1 5]


And plot the obtained matrices:

In [8]:
matrix = processor.matrix_from_data(
    data,
    predictions={"node_labels": node_labels, "edge_labels": edge_labels},
)

plot_basis_matrix(
    matrix[0]*100,
    config1,
    point_lines={"color": "black"},
    basis_lines={"color": "blue"},
    colorscale="temps",
    text=".2f",
    basis_labels=True,
).show()

In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[1] },
  edge_index=[2, 4],
  num_nodes=3,
  neigh_isc=[4],
  n_edges=[1],
  positions=[3, 3],
  shifts=[4, 3],
  cell=[3, 3],
  nsc=[1, 3],
  node_attrs=[3, 2],
  point_types=[3],
  edge_types=[4],
  batch=[3],
  ptr=[2]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x7fe7a85dfe20>
atom_ptr: [0 3]
edge_ptr: [0 4]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0]
  point_block_size = [ 1 25]
  pointers = [ 0  1 26 27]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1]
  edge_block_size = [ 1  5 25]
  pointers = [ 0  5 10 15 20]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:
 this takes the edge labels from edge_labels_ptr[edge_start] =

Using MatrixMACE
----------------

If you don't want to handle the details of interacting `MACE` with `Graph2Mat`, you can also use `MatrixMACE`, which takes a mace model and wraps it to also output the `node_labels` and `edge_labels` corresponding to a matrix. 

Internally, it just initializes a `E3nnGraph2Mat` layer. However it can handle the interaction between `MACE` and `Graph2Mat` in more complex cases like having an extra preprocessing step for edges, which needs some extra inputs from MACE.

In [9]:
from graph2mat.models import MatrixMACE
from graph2mat.bindings.e3nn import E3nnEdgeMessageBlock

In [10]:
matrix_mace_model = MatrixMACE(
    mace_model,
    unique_basis=table,
    readout_per_interaction=True,
    edge_hidden_irreps=o3.Irreps("10x0e + 10x1o + 10x2e"),
    symmetric=False,
)

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/ICN2/snavarro/.local/lib/python3.10/site-packages/torch/ji

The output of this model is MACE's output plus the `node_labels` and `edge_labels` for the predicted matrix:

In [11]:
out = matrix_mace_model(data)

out

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0]
shapes:  [[1 5]
 [1 5]]
shapes_inv:  [[1 5]
 [1 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26  1]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True, False,  True])
j_edges: (~i_edges) tensor([ True, False,  True, False])
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [ 1 -1  1 -1]
shapes:  [[1 1 5]
 [1 5 5]]
shapes_inv:  [[1 1 5]


{'energy': tensor([-0.3206], grad_fn=<SumBackward1>),
 'node_energy': tensor([0., 0., 0.], dtype=torch.float64),
 'contributions': tensor([[ 0.0000,  0.0000, -0.3280, -0.0319,  0.0393]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[-1.5024e-02,  0.0000e+00,  0.0000e+00, -1.9594e-05,  1.4302e-02,
           0.0000e+00,  0.0000e+00,  1.3261e-02, -2.6200e-03],
         [-4.1464e-01,  0.0000e+00,  0.0000e+00, -4.5309e-06, -2.6564e-01,
           0.0000e+00,  0.0000e+00, -4.5035e-05,  2.7080e-02],
         [-1.4325e-02,  0.0000e+00,  0.0000e+00, -3.5213e-05,  1.3495e-02,
           0.0000e+00,  0.0000e+00,  3.0715e-03, -3.2025e-03]],
        grad_fn=<CatBackward0>),
 'node_labels': tensor([ 1.2103e-05,  1.5319e-02, -3.0112e-02,  0.0000e+00,  0.000

You can of course plot the predicted matrices:

In [15]:
matrix = processor.matrix_from_data(data, predictions=out)
from matplotlib import pyplot as plt
plot_basis_matrix(
    matrix[0]*1e5,
    config1,
    point_lines={"color": "black"},
    basis_lines={"color": "blue"},
    colorscale="temps",
    text=".2f",
    basis_labels=True,
).show()


In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[1] },
  edge_index=[2, 4],
  num_nodes=3,
  neigh_isc=[4],
  n_edges=[1],
  positions=[3, 3],
  shifts=[4, 3],
  cell=[3, 3],
  nsc=[1, 3],
  node_attrs=[3, 2],
  point_types=[3],
  edge_types=[4],
  batch=[3],
  ptr=[2]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x7fe7a825ee90>
atom_ptr: [0 3]
edge_ptr: [0 4]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0]
  point_block_size = [ 1 25]
  pointers = [ 0  1 26 27]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1]
  edge_block_size = [ 1  5 25]
  pointers = [ 0  5 10 15 20]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:
 this takes the edge labels from edge_labels_ptr[edge_start] =

In [23]:
print("out edge labels  - obtained from the matrix MACE model and data")
print('Number of edge labels:', len(out['edge_labels']))
print(out['edge_labels']*1e8)

out edge labels  - obtained from the matrix MACE model and data
Number of edge labels: 20
tensor([ -12.1617,  -28.4855,    0.0000,    0.0000,   93.8689,  113.1050,
          74.3992,    0.0000,    0.0000,   16.0304,   76.9095,  -57.0190,
           0.0000,    0.0000,  126.2517,  106.7852, -168.0377,    0.0000,
           0.0000,   20.4792], grad_fn=<MulBackward0>)


# Rotating matrix

A matrix should rotate equivariantly if we rotate the configuration given to mace via the Data. Let us try!

Thing sthat remain constant under rotation: the basis we defined and the thing sthat depend on it.

- table object: an processed object with the basis of each point basis (and the basis points point_1, point_2, point_3, point_4)
- data_processor object processor : has info of how to process the basis, i.e., information about the edges and pointers.
- mace_model and matrix_mace_model : it is just the architecture of the mace model, so we use the same model for both the original and rotated configurations. Matrixmace is just the matrixed version of the mace model, so it is also the same for both configurations.

In [16]:
positions_rot = np.array([[0, 0, 0], [0, 6.0, 0], [0, 9.0, 0]])  # rotated positions

config1_rot = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions_rot,  # changed positions to rotated ones
    basis=[point_1, point_2],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)


dataset_rot = TorchBasisMatrixDataset([config1_rot], data_processor=processor)


loader_rot = DataLoader(dataset_rot, batch_size=1)

data_rot = next(iter(loader_rot))

self.basis_table.R is an array: [2 5]
point_types: [0 1 0]
self.basis_table.R[point_types]: [2 5 2]
Cutoff: [1.9999 4.9999 1.9999]
In BasisMatrixData.from_config 1:
edge_index: [[0 1 1 2]
 [1 0 2 1]]
edge_types: [ 1 -1 -1  1]
In sort_edge_index:
isc_off: [[[0]]]
sc_shifts: [[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
isc: [0 0 0 0]
In BasisMatrixData.from_config 2:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]
In BasisMatrixData.from_config 3:
edge_index: [[0 1 2 1]
 [1 0 1 2]]
edge_types: [ 1 -1  1 -1]


In [17]:
out_rot = matrix_mace_model(data_rot)

out_rot

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0]
shapes:  [[1 5]
 [1 5]]
shapes_inv:  [[1 5]
 [1 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
indices:  [ 0  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26  1]
In Graph2Mat _forward_interactions: 
graph2mat_edge_types:  tensor([ 1, -1,  1, -1], dtype=torch.int32)
edge_types:  tensor([ 1, -1,  1, -1], dtype=torch.int32)
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([ True, False,  True, False])
j_edges: (~i_edges) tensor([False,  True, False,  True])
In Graph2Mat _forward_interactions: 
i_edges: (graph2mat_edge_types[mask] == edge_type)  tensor([False,  True, False,  True])
j_edges: (~i_edges) tensor([ True, False,  True, False])
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [ 1 -1  1 -1]
shapes:  [[1 1 5]
 [1 5 5]]
shapes_inv:  [[1 1 5]


{'energy': tensor([-0.3206], grad_fn=<SumBackward1>),
 'node_energy': tensor([0., 0., 0.], dtype=torch.float64),
 'contributions': tensor([[ 0.0000,  0.0000, -0.3280, -0.0319,  0.0393]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[-1.5024e-02, -1.9594e-05,  0.0000e+00,  0.0000e+00,  1.4302e-02,
           1.3261e-02,  0.0000e+00,  0.0000e+00, -2.6200e-03],
         [-4.1464e-01, -4.5309e-06,  0.0000e+00,  0.0000e+00, -2.6564e-01,
          -4.5035e-05,  0.0000e+00,  0.0000e+00,  2.7080e-02],
         [-1.4325e-02, -3.5213e-05,  0.0000e+00,  0.0000e+00,  1.3495e-02,
           3.0715e-03,  0.0000e+00,  0.0000e+00, -3.2025e-03]],
        grad_fn=<CatBackward0>),
 'node_labels': tensor([ 1.2103e-05,  1.5319e-02, -3.0112e-02, -2.2285e-07,  0.000

You can of course plot the predicted matrices:

In [20]:
matrix_rot = processor.matrix_from_data(data_rot, predictions=out_rot)

plot_basis_matrix(
    matrix_rot[0]*1e5,
    config1_rot,
    point_lines={"color": "black"},
    basis_lines={"color": "blue"},
    colorscale="temps",
    text=".2f",
    basis_labels=True,
).show()

In processing.py matrix_from_data:
data: TorchBasisMatrixDataBatch(
  metadata={ data_processor=[1] },
  edge_index=[2, 4],
  num_nodes=3,
  neigh_isc=[4],
  n_edges=[1],
  positions=[3, 3],
  shifts=[4, 3],
  cell=[3, 3],
  nsc=[1, 3],
  node_attrs=[3, 2],
  point_types=[3],
  edge_types=[4],
  batch=[3],
  ptr=[2]
)
is_batch: True
In MatrixDataProcessor.yield_from_batch:
arrays=data.numpy_arrays(): <graph2mat.core.data.processing.NumpyArraysProvider object at 0x7fe783ed0880>
atom_ptr: [0 3]
edge_ptr: [0 4]
In BasisTableWithEdges.point_block_pointer:
  point_types = [0 1 0]
  point_block_size = [ 1 25]
  pointers = [ 0  1 26 27]
In BasisTableWithEdges.edge_block_pointer:
  edge_types = [ 1 -1  1 -1]
  edge_block_size = [ 1  5 25]
  pointers = [ 0  5 10 15 20]
example 0 (batch):
  atom_start: 0
  atom_end: 3
  edge_start: 0
  edge_end: 4
  new_edge_label = edge_labels[edge_labels_ptr[edge_start]: edge_labels_ptr[edge_end]]:
 this takes the edge labels from edge_labels_ptr[edge_start] =

In [24]:
print("out_rot edge labels  - obtained from the matrix MACE model and data")
print('Number of edge labels:', len(out_rot['edge_labels']))
print(out_rot['edge_labels']*1e8)

out_rot edge labels  - obtained from the matrix MACE model and data
Number of edge labels: 20
tensor([ -12.1617,  -28.4855,   93.8689,    0.0000,    0.0000,  113.1050,
          74.3992,   16.0304,    0.0000,    0.0000,   76.9095,  -57.0190,
         126.2517,    0.0000,    0.0000,  106.7852, -168.0377,   20.4792,
           0.0000,    0.0000], grad_fn=<MulBackward0>)


Summary and next steps
----------------------

In this notebook we learned **how to interface MACE with Graph2Mat**.

The **next steps** could be:

- **Train a MACE+Graph2Mat model** following the steps in [this notebook](<./Fitting matrices.ipynb>), replacing the model by the `MACE+Graph2Mat` model.